# Loading Data into EpiLearn

Every task in EpiLearn is fed by a `Dataset` object (called `UniversalDataset` before
0.1.0 -- the old name still imports, but `Dataset` is the name to use). This notebook
walks through the four ways to build one:

1. by name, from a built-in dataset,
2. from the bundled toy dataset,
3. from your own tensors / numpy arrays,
4. from CSV files (long format + an optional edge list).

In [1]:
import os
import torch
import numpy as np

# The paths below are relative to the repository root, so step out of examples/.
# (Guarded so that re-running the cell is harmless.)
if os.path.basename(os.getcwd()) == "examples":
    os.chdir("..")
print("working directory:", os.getcwd())

from epilearn.data import Dataset

working directory: /home/ec2-user/EpiLearn_public


/home/ec2-user/miniconda3/envs/epilearn/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Built-in datasets

`Dataset(name=..., root=...)` downloads the named dataset into `root` the first time and
reads it from there afterwards. `Covid_<Country>` gives a spatiotemporal dataset: a
feature tensor of shape `(time, region, feature)` plus a static contact graph.
Supported countries: China, Brazil, Austria, England, France, Italy, NewZealand, Spain.

In [2]:
dataset = Dataset(name='Covid_Austria', root='./datasets/')

print("x            :", tuple(dataset.x.shape), "(time, regions, features)")
print("graph        :", tuple(dataset.graph.shape))
print("feature_names:", dataset.feature_names)   # renamed from `.features` in 0.1.0
print("timestamps   :", dataset.timestamps[:3], "...")
print("n_timesteps  :", dataset.n_timesteps, "| n_regions:", dataset.n_regions,
      "| n_features:", dataset.n_features)

x            : (166, 9, 3) (time, regions, features)
graph        : (9, 9)
feature_names: ['infect', 'recover', 'death']
timestamps   : ['X2020.07.31.csv', 'X2020.07.25.csv', 'X2020.07.19.csv'] ...
n_timesteps  : 166 | n_regions: 9 | n_features: 3


`Measles` is a temporal-only dataset: 946 English towns x 1108 bi-weekly counts, stored
as one row per town. Extra information that came with the file (populations, births,
coordinates) lives in `dataset.metadata` since 0.1.0.

In [3]:
measles = Dataset(name='Measles', root='./datasets/')

print("x        :", tuple(measles.x.shape), "(towns, bi-weeks)")
print("metadata :", list(measles.metadata.keys()))
print("first ten towns:", measles.feature_names[:10])

x        : (946, 1108) (towns, bi-weeks)
metadata : ['anual_population', 'anual_birth', 'coordinates']
first ten towns: ['Abingdon', 'Abram', 'Accrington', 'Acton', 'Adlington', 'Adwick.le.Street', 'Aldeburgh', 'Alderley.Edge', 'Aldershot', 'Aldridge']


To forecast a single town, slice out its series and wrap it in a fresh `Dataset`.
EpiLearn treats a dataset as temporal when `x` has two dimensions `(time, feature)`
and as spatiotemporal when it has three `(time, region, feature)`.

In [4]:
town = measles.feature_names[100]
series = measles.x[100].unsqueeze(1)          # (1108,) -> (1108, 1)

one_town = Dataset(x=series, y=series.clone(), feature_names=[town], target_names=[town])
print(town, "->", tuple(one_town.x.shape), "| spatiotemporal:", one_town.is_spatiotemporal)

Boston -> (1108, 1) | spatiotemporal: False


## 2. The toy dataset

`load_toy_dataset()` downloads a small spatiotemporal example (47 regions, 539 steps)
into `./datasets`. It is the dataset used by the quick-start snippets in the README, and
it is the only built-in that also carries states (SIR compartments) and a dynamic graph.

In [5]:
dataset = Dataset()
dataset.load_toy_dataset()

print("x            :", tuple(dataset.x.shape), "(time, regions, features)")
print("y            :", tuple(dataset.y.shape))
print("states       :", tuple(dataset.states.shape), "(S, I, R per region)")
print("graph        :", tuple(dataset.graph.shape))
print("dynamic_graph:", tuple(dataset.dynamic_graph.shape))
print("edge_index   :", tuple(dataset.edge_index.shape))
print("edge_weight  :", tuple(dataset.edge_weight.shape))
print(dataset)

x            : (539, 47, 4) (time, regions, features)
y            : (539, 47)
states       : (539, 47, 3) (S, I, R per region)
graph        : (47, 47)
dynamic_graph: (539, 47, 47, 1)
edge_index   : (2, 2189)
edge_weight  : (2189,)
Dataset(Spatiotemporal, x=(539, 47, 4), y=(539, 47), graph=(47, 47))


### Saving and reloading

`Dataset.save()` takes an explicit `path` in 0.1.0 (it used to write to a fixed
location), and `Dataset.load()` reads it back.

In [6]:
import tempfile

path = os.path.join(tempfile.gettempdir(), "toy_dataset.pt")
dataset.save(path)

reloaded = Dataset.load(path)
print("reloaded x:", tuple(reloaded.x.shape), "| graph:", tuple(reloaded.graph.shape))

reloaded x: (539, 47, 4) | graph: (47, 47)


## 3. From your own arrays

Anything you can get into a numpy array or a torch tensor can become a `Dataset`.
Here we read the raw `.npy` behind the toy dataset and rebuild the object by hand: `x`
is `(time, region, feature)`, `y` is `(time, region)`, and `graph` is a dense adjacency.

In [7]:
raw = np.load('datasets/features.npy', allow_pickle=True).tolist()
print("keys in the raw file:", list(raw.keys()))

x = torch.FloatTensor(raw['node'])            # (time, regions, features)
adj = torch.FloatTensor(np.load('datasets/graphs.npy'))

custom = Dataset(x=x, y=x[:, :, 0], graph=adj, states=torch.FloatTensor(raw['SIR']))
print("x:", tuple(custom.x.shape), "| y:", tuple(custom.y.shape),
      "| graph:", tuple(custom.graph.shape))

keys in the raw file: ['od', 'node', 'SIR']
x: (539, 47, 4) | y: (539, 47) | graph: (47, 47)


## 4. From CSV

`Dataset.from_csv` reads long-format data: one row per (timestamp, region). Its keyword
names changed in 0.1.0 -- `file_path`, `timestamp_col`, `region_col`, `graph_file`
(previously `feature_csv`, `time_col`, `node_id_col`, `edge_csv`). Omit `region_col`
for a single time series.

In [8]:
csv_dataset = Dataset.from_csv(
    file_path='datasets/toy_features.csv',
    timestamp_col='time',
    region_col='node',
    feature_cols=['f0', 'f1', 'f2', 'f3'],
    target_cols=['y'],
    graph_file='datasets/toy_edges.csv',
)

print("x           :", tuple(csv_dataset.x.shape), "(time, regions, features)")
print("y           :", tuple(csv_dataset.y.shape))
print("graph       :", tuple(csv_dataset.graph.shape))
print("features    :", csv_dataset.feature_names, "-> target:", csv_dataset.target_names)
print("regions     :", csv_dataset.regions[:5], "...")

x           : (539, 47, 4) (time, regions, features)
y           : (539, 47)
graph       : (47, 47)
features    : ['f0', 'f1', 'f2', 'f3'] -> target: ['y']
regions     : [0, 1, 2, 3, 4] ...


## Where to go next

Any of these datasets can be handed straight to a task:

```python
from epilearn.tasks import Forecast
task = Forecast(prototype=GRUModel, lookback=12, horizon=3, device='cpu')
result = task.rolling_train(dataset, train_size=300, val_size=60, test_size=60)
```

See `forecast_task.ipynb` for forecasting, `detection_task.ipynb` for source detection,
and `tutorial.ipynb` for the guided walk-through.